In [ ]:
cd ../..

In [ ]:
import yaml, sys
import pandas as pd
import numpy as np
import plotly.express as px
from src.feature_importance import FeatureImportance
from src.reduce_dimensions import ReduceDimensions
from loguru import logger
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

sns.set_context("poster")
sns.set_style("ticks")

np.random.seed(0)

logger.remove()
logger.add(sys.stderr, level="WARNING")

# Old

In [ ]:
methods = [
    ("triu", "FR-RSA"),
    ("cholesky", "MLEM cholesky"),
    ("exp", "MLEM exp"),
]
max_epochs = [1, 5, 10, 20, 30, 50, 100, 200, 500, 1000]
all_importances = []
all_spearman = []
with tqdm(total=len(methods) * len(max_epochs)) as pbar:
    for param, method in methods:
        for layer in layers:
            for eps in epsilons:
                cfg = f"""
                dataset:
                    path: datasets/relative_clause.csv
                trainer:
                    max_epochs: 1000
                    dataloader_builder:
                        cv: 3
                    representations:
                        model_name: bert-base-uncased
                        layer: {layer}
                    model_builder:
                        param: {param}
                    eps: {eps}
                """
                cfg = yaml.safe_load(cfg)
                fi = FeatureImportance(**cfg)
                i, s, w = fi.compute()
                for e in [i, s]:
                    e["Method"] = method
                    e["Layer"] = layer
                    e["Epsilon"] = f"Epsilon {eps:.0e}"
                all_importances.append(i)
                s["variable"] = "Encoding Spearman"
                all_spearman.append(s.copy())
                s["variable"] = "Training duration (s)"
                s["mean"] = w.training_duration.iloc[0]
                all_spearman.append(s.copy())
                s["variable"] = "Converged?"
                s["mean"] = w.converged.iloc[0]
                all_spearman.append(s.copy())
                s["variable"] = "n_epochs"
                s["mean"] = w.n_epochs.iloc[0]
                all_spearman.append(s)
                pbar.update(1)
all_importances = pd.concat(all_importances)
all_spearman = pd.concat(all_spearman)

In [ ]:
all_importances = all_importances[all_importances.split == "test"]
all_spearman = all_spearman[all_spearman.split == "test"]

In [ ]:
features = (
    all_importances.sort_values("mean", ascending=False)
    .groupby("Method")
    .Feature.apply(lambda x: x.unique()[:5])
    .reset_index()
)
features = features.explode("Feature")

In [ ]:
g = sns.relplot(
    all_importances.merge(features),
    x="Layer",
    y="mean",
    hue="Feature",
    aspect=1.5,
    row="Method",
    col="Epsilon",
    kind="line",
)
g.set_titles(col_template="{col_name}", row_template="{row_name}")

In [ ]:
g = sns.relplot(
    all_spearman,
    x="Layer",
    y="mean",
    aspect=1.5,
    hue="Method",
    col="Epsilon",
    row="variable",
    kind="line",
    facet_kws={"sharey": False},
)
g.set_titles(col_template="{col_name}", row_template="{row_name}")

# Parallel

In [ ]:
infra_gpu = """
folder: .cache
cluster: auto
mode: retry
cpus_per_task: 24
gpus_per_node: 1
timeout_min: 120
slurm_qos: qos_gpu_h100-dev
slurm_constraint: h100
slurm_account: ioj@h100
slurm_additional_parameters:
    hint: nomultithread
"""
infra_gpu = yaml.safe_load(infra_gpu)
infra_cpu = """
folder: .cache
mode: retry
cluster: auto
cpus_per_task: 8
timeout_min: 120
slurm_account: ioj@cpu
slurm_additional_parameters:
    hint: nomultithread
"""
infra_cpu = yaml.safe_load(infra_cpu)

In [ ]:
def get_cfg(method, max_epochs):
    cfg = f"""
    dataset:
        path: datasets/relative_clause.csv
    trainer:
        dataloader_builder:
            cv: 5
        model_builder:
            param: {method}
        max_epochs: {max_epochs}
        representations:
            model_name: bert-base-uncased
            layer: 7
        eps: 1e-5
    """
    cfg = yaml.safe_load(cfg)
    cfg["infra"] = infra_cpu

    return cfg

In [ ]:
methods = [
    ("triu", "FR-RSA"),
    ("cholesky", "MLEM cholesky"),
    ("exp", "MLEM exp"),
]
max_epochs = [1, 5, 10, 20, 30, 50, 100, 200, 300, 500, 750, 1000]

# Launch first task to ensure precomputed latents
cfg = get_cfg(methods[0][0], max_epochs[0])
cfg["infra"] = infra_gpu
fi = FeatureImportance(**cfg)
i, s, w = fi.compute()

In [ ]:
cfg = get_cfg(methods[0][0], max_epochs[0])
fi = FeatureImportance(**cfg)
results = []
with tqdm(total=len(methods) * len(max_epochs), desc="Creating tasks") as pbar:
    with fi.infra.job_array() as array:
        for param, _ in methods:
            for k in max_epochs:
                task_to_compute = fi.infra.clone_obj(
                    {"trainer": {"max_epochs": k, "model_builder": {"param": param}}}
                )
                array.append(task_to_compute)
                results.append([param, k, task_to_compute])
                pbar.update(1)

In [ ]:
importances, spearman, weights = [], [], []
for param, k, fi in tqdm(results):
    i, s, w = fi.compute()
    for e in [i, s, w]:
        e["Method"] = param
        e["max_epochs"] = k
    importances.append(i)
    spearman.append(s)
    weights.append(w)
pd.concat(importances).to_parquet("experiments/fr_rsa/importances.parquet")
pd.concat(spearman).to_parquet("experiments/fr_rsa/spearman.parquet")
pd.concat(weights).to_parquet("experiments/fr_rsa/weights.parquet")

## Viz

In [ ]:
importances = pd.read_parquet("experiments/fr_rsa/importances.parquet")
spearman = pd.read_parquet("experiments/fr_rsa/spearman.parquet")
weights = pd.read_parquet("experiments/fr_rsa/weights.parquet")

In [ ]:
sns.lineplot(
    spearman.query("split == 'test'"),
    x="max_epochs",
    y="mean",
    hue="Method",
    markers=True,
)
plt.xscale("log")